# Cost, emissions and water-scarcity trade-offs

This notebook explores the trade-offs that GLADE's **AWARE2.0 water module** opens
up between the economic cost of the food system, its greenhouse-gas emissions, and
its impact on **water scarcity** -- and how those trade-offs change when the diet is
allowed to adapt.

## The water-scarcity signal

AWARE2.0 (Seitfudem et al. 2025, built on WaterGAP2.2e) assigns every river basin a
*characterisation factor* (CF, in m3 world-equivalent per m3) measuring how scarce a
unit of consumed water is there. The published CF is **marginal** -- the impact of
*one more* cubic metre at today's operating point -- which is invalid for a model that
re-decides *all* irrigation. GLADE therefore reconstructs the **non-marginal** curve:
as the model draws down a basin's agricultural pool its remaining water shrinks and its
CF rises. That convex curve is discretised into a **tiered water supply**, ordered from
abundant (low-CF) to stressed (high-CF) water. Because the water itself is a free input,
a negligible **merit-order regularizer** (a tiny per-tier cost proportional to CF) makes
drawing the least-scarce water the *unpriced* optimum rather than an arbitrary tie -- so
the baseline is the merit-order minimum, not a degenerate high-CF draw. Total accumulated
scarcity is tracked in a global store that can be **priced** or **capped** at solve time,
exactly like the GHG store.

## The experiment

We sweep two policy levers --

* a **water-scarcity price** (USD per m3 world-equivalent), and
* a **carbon price** (USD per tCO2e) --

under **two diet regimes**, each anchored to the 2020 baseline so that without any
externality pricing the model reproduces present-day consumption:

* **Fixed diet** (`water_tradeoff_fixed_diet`): consumption is pinned to the 2020
  baseline, so the only way to cut water scarcity or emissions is on the **production
  side** -- where and how crops are grown, irrigated-vs-rainfed sourcing, and basin
  reallocation.
* **Flexible diet** (`water_tradeoff_flexible_diet`): consumption can re-optimise via
  calibrated piecewise food-utility blocks (per-country calorie intake held at
  baseline), so impacts can also be cut by **shifting diet composition** away from
  water- and emission-intensive foods.

Comparing the two isolates whether **diet flexibility** helps reduce water scarcity.
Sweeping each lever traces the relevant front; combining them exposes the water-carbon
interaction. All three outcomes -- cost, emissions, total water scarcity -- are read
back from every solved scenario.

> Resolution: full default build (750 regions, 3 resource classes, 48 crops, 20 trade
> hubs), solved with Gurobi. Costs and production-stability penalties are calibrated
> against the same AWARE water supply the scenarios solve under.

In [ ]:
import logging
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import yaml

warnings.filterwarnings("ignore")
logging.getLogger("pypsa").setLevel(logging.WARNING)  # silence per-network import logs

# Locate the repository root (works whether run from repo root or notebooks/).
ROOT = Path.cwd()
while (
    not (ROOT / "config" / "water_tradeoff_fixed_diet.yaml").exists()
    and ROOT.parent != ROOT
):
    ROOT = ROOT.parent

# Two diet regimes, each its own config + results directory. The unpriced
# reference scenario differs (fixed diet: "default"; flexible diet: "reference"),
# and the flexible config carries a calibration-only "baseline" scenario (used to
# extract consumer values) that is excluded from the trade-off fronts.
REGIMES = {
    "fixed": {
        "config": "water_tradeoff_fixed_diet",
        "reference": "default",
        "exclude": set(),
    },
    "flexible": {
        "config": "water_tradeoff_flexible_diet",
        "reference": "reference",
        "exclude": {"baseline"},
    },
}


def lever_prices(scenarios, name):
    d = scenarios[name]
    em, wa = d.get("emissions", {}), d.get("water_scarcity", {})
    ghg = em.get("ghg_price", 0) if em.get("ghg_pricing_enabled", False) else 0
    water = wa.get("price", 0) if wa.get("pricing_enabled", False) else 0
    return water, ghg


def load_metrics(config, scenarios, name):
    n = pypsa.Network(
        str(ROOT / "results" / config / "solved" / f"model_scen-{name}.nc")
    )
    e = n.stores.dynamic.e.iloc[0]
    water_impact = float(e["store:impact:water_scarcity"]) / 1e6  # millions of Mm3-eq
    ghg = float(e["store:emission:ghg"])  # MtCO2e
    wp, gp = lever_prices(scenarios, name)
    # Objective minus the externality shadow-price terms = real system cost.
    cost = float(n.objective) - wp * 1e-3 * water_impact * 1e6 - gp * 1e-3 * ghg
    links = n.links.static
    flow = n.links.dynamic.p0.iloc[0]
    tiers = links[links["carrier"] == "water_supply"]
    drawn = flow[tiers.index].clip(lower=0)
    total = float(drawn.sum())
    mean_cf = float((tiers["efficiency2"] * drawn).sum() / total) if total else np.nan
    cf_draw = pd.DataFrame(
        {"cf": tiers["efficiency2"].to_numpy(), "draw": drawn.to_numpy()}
    )
    return {
        "diet": None,  # filled by caller
        "scenario": name,
        "water_price": wp,
        "carbon_price": gp,
        "cost_bn": cost,
        "water_scarcity": water_impact,
        "emissions": ghg,
        "withdrawn_Mm3": total,
        "mean_cf": mean_cf,
    }, cf_draw


records, cf_draws = [], {}
for diet, spec in REGIMES.items():
    cfg = yaml.safe_load((ROOT / "config" / f"{spec['config']}.yaml").read_text())
    # Drop suppressed scenarios (value None, e.g. the implicit "default").
    scenarios = {k: v for k, v in cfg["scenarios"].items() if v is not None}
    for name in scenarios:
        # Skip the no-deviation-penalty ('nd') scenarios here; they are
        # analysed separately in the dev-vs-nodev section below.
        if name in spec["exclude"] or name.endswith("nd"):
            continue
        rec, cf = load_metrics(spec["config"], scenarios, name)
        rec["diet"] = diet
        records.append(rec)
        cf_draws[(diet, name)] = cf

df = pd.DataFrame(records).set_index(["diet", "scenario"])
# Per-regime unpriced reference cost (for cost-premium fronts) and the reference
# scenario name, kept handy for the plots below.
REF = {diet: spec["reference"] for diet, spec in REGIMES.items()}
df.round(2)

## Reading the table

The table is indexed by (`diet`, `scenario`). Within each diet regime:

* `water_scarcity` is in **millions of Mm3 world-equivalent**; `emissions` in **MtCO2e**
  (negative = net sequestration via spared land); `withdrawn_Mm3` is total irrigation
  drawn; `mean_cf` is the volume-weighted characterisation factor of that water -- a direct
  read-out of *which* basins are tapped.
* `cost_bn` is the real system cost with the policy shadow-prices removed. **Absolute cost
  is not comparable across diet regimes** (the flexible objective also carries
  consumer-utility terms), so all cost comparisons below use the *premium relative to each
  regime's own unpriced reference* (`default` for fixed, `reference` for flexible).

The `wp*` rows price water at zero carbon price; `ghg*` price carbon at zero water price;
`ghg200wp*` combine a \$200/t carbon price with a water price.

In [ ]:
plt.rcParams.update(
    {
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# One style per diet regime; fronts are always taken relative to each regime's
# own unpriced reference (absolute cost is not comparable across regimes, since
# the flexible objective also carries consumer-utility terms).
DIET_STYLE = {
    "fixed": {"color": "#1f77b4", "marker": "o", "ls": "-", "label": "fixed diet"},
    "flexible": {
        "color": "#d62728",
        "marker": "s",
        "ls": "--",
        "label": "flexible diet",
    },
}


def fronts(diet):
    sub = df.loc[diet]
    refcost = sub.loc[REF[diet], "cost_bn"]
    wf = sub[sub.carbon_price == 0].sort_values(
        "water_price"
    )  # water front, carbon off
    gf = sub[sub.water_price == 0].sort_values(
        "carbon_price"
    )  # carbon front, water off
    return sub, refcost, wf, gf


fig, ax = plt.subplots(2, 2, figsize=(12, 9))

for diet, st in DIET_STYLE.items():
    sub, refcost, wf, gf = fronts(diet)
    line = {"color": st["color"], "marker": st["marker"], "ls": st["ls"]}

    # A: cost premium vs water scarcity (each regime vs its own unpriced ref)
    ax[0, 0].plot(wf.water_scarcity, wf.cost_bn - refcost, label=st["label"], **line)
    # B: mean CF of withdrawn water vs water price (priced points only)
    wfp = wf[wf.water_price > 0]
    ax[0, 1].plot(wfp.water_price, wfp.mean_cf, label=st["label"], **line)
    # C: carbon abatement front (water unpriced)
    ax[1, 0].plot(gf.emissions, gf.cost_bn - refcost, label=st["label"], **line)
    # D: emissions co-benefit of water pricing
    ax[1, 1].plot(wf.water_scarcity, wf.emissions, label=st["label"], **line)

# --- Panel A ---
a = ax[0, 0]
a.set_xscale("log")
a.invert_xaxis()
a.set(
    xlabel="Total water scarcity (Mm3 world-eq, millions)",
    ylabel="Added cost vs unpriced (bn USD)",
    title="A. Cost of reducing water scarcity",
)
a.legend(frameon=False, title="diet regime")
wf_fix = df.loc["fixed"]
a.annotate(
    "both diets start at the same\nmerit-order baseline (~{:.1f})".format(
        wf_fix.loc["default", "water_scarcity"]
    ),
    xy=(wf_fix.loc["default", "water_scarcity"], 0.4),
    xytext=(2.7, 32),
    arrowprops={"arrowstyle": "->", "color": "gray"},
    fontsize=8.5,
    color="gray",
)
a.annotate(
    "deep cuts are genuinely costly\n(less irrigation, production shifts):\n"
    "~${:.0f} bn for {:.1f} -> {:.1f}".format(
        wf_fix.loc["wp3", "cost_bn"] - wf_fix.loc["default", "cost_bn"],
        wf_fix.loc["default", "water_scarcity"],
        wf_fix.loc["wp3", "water_scarcity"],
    ),
    xy=(
        wf_fix.loc["wp3", "water_scarcity"],
        wf_fix.loc["wp3", "cost_bn"] - wf_fix.loc["default", "cost_bn"],
    ),
    xytext=(0.85, 55),
    arrowprops={"arrowstyle": "->", "color": "gray"},
    fontsize=8.5,
    color="gray",
)

# --- Panel B ---
b = ax[0, 1]
b.set_xscale("log")
b.set(
    xlabel="Water-scarcity price (USD / m3 world-eq)",
    ylabel="Mean characterisation factor (m3-eq/m3)",
    title="B. Higher prices draw down basins (less irrigation)",
)
b.legend(frameon=False, fontsize=9)
for diet, st in DIET_STYLE.items():
    b.axhline(
        df.loc[(diet, REF[diet]), "mean_cf"], ls=":", color=st["color"], alpha=0.5
    )

# --- Panel C ---
c = ax[1, 0]
c.set(
    xlabel="Net GHG emissions (MtCO2e)",
    ylabel="Cost premium vs own reference (bn USD)",
    title="C. Carbon abatement front (water unpriced)",
)
c.invert_xaxis()
c.legend(frameon=False, fontsize=9)
for _s, r in df.loc["fixed"].pipe(lambda s: s[s.water_price == 0]).iterrows():
    if r.carbon_price > 0:
        c.annotate(
            f"${r.carbon_price:.0f}/t",
            (r.emissions, r.cost_bn - df.loc[("fixed", REF["fixed"]), "cost_bn"]),
            fontsize=7.5,
            textcoords="offset points",
            xytext=(4, 3),
            color="gray",
        )

# --- Panel D ---
d = ax[1, 1]
d.set_xscale("log")
d.invert_xaxis()
d.set(
    xlabel="Total water scarcity (Mm3 world-eq, millions)",
    ylabel="Net GHG emissions (MtCO2e)",
    title="D. Co-benefit: pricing water also cuts emissions",
)
d.legend(frameon=False, fontsize=9)

fig.suptitle(
    "Cost / emissions / water-scarcity trade-offs: fixed vs flexible diet "
    "(AWARE2.0 tiered water supply)",
    fontsize=13,
    y=0.99,
)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## What the fronts say

**Panel A -- there is no cheap water-scarcity reduction; the baseline is already
merit-ordered.** With the merit-order regularizer in place, the unpriced optimum
already draws the least-scarce water available within each region (mean CF ~5.9,
total scarcity ~3.4), so the faintest prices (`wp0003`, `wp001`) do essentially
nothing. Reducing scarcity *below* that baseline requires meaningful prices
(>= ~0.003) and follows a convex cost tail: cutting scarcity from ~3.4 to ~0.3
costs ~$90 bn. Crucially -- unlike the degenerate result this replaces -- the
reduction now comes from genuinely using *less* irrigation (panel B: withdrawn
volume falls from ~583 to ~400) and shifting production, which is exactly where
the production-stability penalties and real trade-offs bite.

**Diet flexibility barely moves the water front at moderate prices.** The fixed and
flexible curves in panel A nearly coincide near the baseline, and both start from the
*same* baseline scarcity (~3.4): an earlier apparent "flexible diet starts lower" gap
was an artifact of the unpriced degeneracy, not a real effect. The water lever works
through where and how crops are grown (basin drawdown, irrigated->rainfed), which is
available regardless of whether the diet can change. Diet flexibility only pulls
clearly ahead at *high* prices, when eliminating the last irrigation is much cheaper if
the diet can shift off thirsty foods (see "How much would eliminating irrigation cost?"
below).

**Panel C** is the carbon-abatement curve: higher carbon prices buy deep net-negative
emissions (land sparing and sequestration) at rising cost, under both diets.

**Panel D -- water and carbon goals are aligned, and here diet flexibility *does*
matter.** Pricing water also lowers emissions (less irrigation spares land, which
sequesters carbon). The co-benefit is several times larger under the flexible diet
(~-190 vs ~-40 MtCO2e at the strongest water price), because a flexible diet can shift
away from foods that are both water- and emission-intensive -- the one place diet
adaptation meaningfully helps in this experiment.

## The mechanism, directly

The figure below bins the water actually withdrawn by the characterisation factor of
its source basin. With the merit-order baseline, even the *unpriced* solution already
draws almost entirely low-CF (abundant) water -- the high-CF (stressed) bands are
negligible from the start. Raising the water price then cuts the *total* withdrawn
volume (genuinely using less irrigation), rather than re-sourcing, since there is
little high-CF water left to avoid.

In [ ]:
# Mechanism shown for the fixed diet (production-side only); the flexible diet
# behaves the same way -- the draw pattern is diet-independent.
DIET_SHOWN = "fixed"
bands = [0.1, 0.5, 1, 2, 5, 20, 100.01]
band_labels = ["0.1-0.5", "0.5-1", "1-2", "2-5", "5-20", "20-100"]
sel = [(DIET_SHOWN, s) for s in ["default", "wp0003", "wp003", "wp03", "wp3"]]
sel_labels = ["unpriced", "$0.0003", "$0.003", "$0.03", "$0.30"]

mat = pd.DataFrame(
    [
        cf_draws[key]
        .assign(
            band=pd.cut(
                cf_draws[key]["cf"], bands, labels=band_labels, include_lowest=True
            )
        )
        .groupby("band", observed=False)["draw"]
        .sum()
        / 1e3
        for key in sel
    ],
    index=sel_labels,
)

fig2, ax2 = plt.subplots(figsize=(9, 5))
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, len(band_labels)))
bottom = np.zeros(len(sel))
for j, bl in enumerate(band_labels):
    ax2.bar(sel_labels, mat[bl], bottom=bottom, color=colors[j], label=bl, width=0.7)
    bottom += mat[bl].to_numpy()
ax2.set(
    xlabel="Water-scarcity price (USD / m3 world-eq)",
    ylabel="Water withdrawn (Mm3 x1000)",
    title=f"Withdrawn irrigation water by basin CF ({DIET_SHOWN} diet)",
)
ax2.legend(
    title="CF band (m3-eq/m3)",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    frameon=False,
)
ax2.text(
    0.97,
    0.78,
    "even unpriced, withdrawals are\nalready low-CF (merit order);\n"
    "higher prices cut the total volume",
    transform=ax2.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    color="dimgray",
)
fig2.tight_layout()
plt.show()

## How much would eliminating irrigation cost?

Extending the water-price sweep to high prices (up to 100 USD/m3-world-eq) pushes
irrigation toward zero. There is **no clean cutoff**: withdrawn volume approaches zero
*asymptotically* along a steep convex cost curve, with no hard floor or infeasibility --
the model can run the fixed 2020 diet almost entirely rainfed, just at enormous cost.
Reaching near-total elimination (~99.5% of irrigation removed) costs of order **several
hundred billion USD/yr** above the unpriced baseline. This is the one place diet
flexibility clearly pays off on water: a flexible diet reaches lower irrigation at every
price and eliminates it appreciably more cheaply (~25% less at the high end), by
substituting away from the thirstiest foods.

> Note: irrigated *area* floors around ~100 Mha even as water *draw* approaches zero
> (those crops stay irrigation-classified but draw negligible water), so the
> scarcity-relevant metric here is withdrawn volume, not irrigated area.

In [ ]:
# Pushing irrigation toward zero: the water-price sweep extends to high prices
# (up to 100 USD/m3-world-eq). Left: withdrawn volume as % of the unpriced
# baseline vs price. Right: the cost of that elimination (premium vs own
# reference) against the irrigation that remains.
fig3, ax3 = plt.subplots(1, 2, figsize=(12, 4.5))
for diet, st in DIET_STYLE.items():
    _, refcost, wf, _ = fronts(diet)
    wf = wf.sort_values("water_price")
    pct = 100 * wf["withdrawn_Mm3"] / wf["withdrawn_Mm3"].iloc[0]  # iloc[0] = unpriced
    prem = wf["cost_bn"] - refcost
    line = {"color": st["color"], "marker": st["marker"], "ls": st["ls"]}
    priced = wf["water_price"] > 0
    ax3[0].plot(wf["water_price"][priced], pct[priced], label=st["label"], **line)
    ax3[1].plot(pct, prem, label=st["label"], **line)

ax3[0].set_xscale("log")
ax3[0].set(
    xlabel="Water-scarcity price (USD / m3 world-eq)",
    ylabel="Irrigation withdrawn (% of unpriced)",
    title="Driving irrigation toward zero",
)
ax3[0].legend(frameon=False)
ax3[1].invert_xaxis()
ax3[1].set(
    xlabel="Irrigation withdrawn (% of unpriced)",
    ylabel="Added cost vs unpriced (bn USD)",
    title="Cost of (nearly) eliminating irrigation",
)
ax3[1].legend(frameon=False, loc="upper left")
_fc = (
    df.loc[("fixed", "wp1000"), "cost_bn"] - df.loc[("fixed", REF["fixed"]), "cost_bn"]
)
_xc = (
    df.loc[("flexible", "wp1000"), "cost_bn"]
    - df.loc[("flexible", REF["flexible"]), "cost_bn"]
)
ax3[1].text(
    0.97,
    0.05,
    "near-total elimination (~99.5%):\n"
    f"fixed ~${_fc:.0f} bn, flexible ~${_xc:.0f} bn\n"
    "(asymptotic -- no clean cutoff)",
    transform=ax3[1].transAxes,
    ha="right",
    va="bottom",
    fontsize=8.5,
    color="dimgray",
)
fig3.tight_layout()
plt.show()

## Why does pricing water *reduce* emissions? The land-use mechanism

Panel D showed a co-benefit that is, at first sight, backwards. Irrigated crops are
high-yield and therefore *land-efficient*, so a naive expectation is that cutting
irrigation should push production onto more (lower-yield) rainfed land, expand the
cropland footprint, drive deforestation, and *raise* emissions. Instead emissions
*fall* -- and several times more under a flexible diet. The decomposition below shows
why: the effect is not a land-footprint story at all.

We walk the water-price front (carbon off) and split the response into (a) net
emissions by source and (b) the physical land-use response (cropland, spared land,
land converted, irrigation draw).

In [ ]:
# Decompose the water-pricing emissions co-benefit into source-level and land-use terms.
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from workflow.scripts.analysis.extract_net_emissions import extract_net_emissions

CH4_GWP, N2O_GWP = 27.0, 273.0  # AR6 GWP100, matching config/default.yaml
FRONT = ["wp003", "wp03", "wp3"]  # priced points along the water front


def _load(config, scen):
    n = pypsa.Network(
        str(ROOT / "results" / config / "solved" / f"model_scen-{scen}.nc")
    )
    e = extract_net_emissions(n, CH4_GWP, N2O_GWP).groupby("source")["mtco2eq"].sum()
    return n, e


def _land(n):
    """Physical land-use read-outs (Mha; irrigation draw in Gm3)."""
    links, p0 = n.links.static, n.links.dynamic.p0.iloc[0]
    cp = links[links["carrier"].isin(["crop_production", "crop_production_multi"])]
    area = p0.reindex(cp.index).clip(lower=0)

    def pick(mask):
        return float(p0.reindex(links.index[mask]).clip(lower=0).sum())

    return {
        "cropland (Mha)": float(area.sum()),
        "irrig. crop area (Mha)": float(area[cp["water_supply"] == "irrigated"].sum()),
        "spared land (Mha)": pick(
            links["carrier"].isin(["spare_land", "spare_existing_grassland"])
        ),
        "land converted (Mha)": pick(links["carrier"] == "land_conversion"),
        "irrigation draw (Gm3)": pick(links["carrier"] == "water_supply") / 1e3,
    }


ems, lands = {}, {}
for diet, spec in REGIMES.items():
    for scen in [REF[diet], *FRONT]:
        n, e = _load(spec["config"], scen)
        key = "ref" if scen == REF[diet] else scen
        ems[(diet, key)], lands[(diet, key)] = e, _land(n)

# Source-level change at the strongest priced point (wp3) vs each regime's reference.
SOURCES = [
    "Enteric fermentation",
    "Rice cultivation",
    "Manure: managed systems",
    "Synthetic fertilizer application",
    "Manure: pasture deposition",
    "Crop residue incorporation",
    "Land Use Change",
    "Carbon sequestration",
]
delta = pd.DataFrame(
    {
        diet: ems[(diet, "wp3")].reindex(SOURCES) - ems[(diet, "ref")].reindex(SOURCES)
        for diet in REGIMES
    }
)

fig5, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: which emission sources move (wp3 vs reference), fixed vs flexible.
y, h = np.arange(len(SOURCES)), 0.38
ax_a.barh(
    y + h / 2,
    delta["flexible"],
    height=h,
    color=DIET_STYLE["flexible"]["color"],
    label="flexible diet",
)
ax_a.barh(
    y - h / 2,
    delta["fixed"],
    height=h,
    color=DIET_STYLE["fixed"]["color"],
    label="fixed diet",
)
ax_a.axvline(0, color="k", lw=0.8)
ax_a.set_yticks(y)
ax_a.set_yticklabels(SOURCES, fontsize=9)
ax_a.invert_yaxis()
ax_a.set(
    xlabel="Change in net emissions at wp3 (MtCO2e)",
    title="A. What drives the emissions drop",
)
ax_a.legend(frameon=False, fontsize=9, loc="lower left")
for diet in REGIMES:
    tot = delta[diet].sum()
    ax_a.annotate(
        f"{diet}: net {tot:+.0f} MtCO2e",
        xy=(0.98, 0.92 if diet == "flexible" else 0.84),
        xycoords="axes fraction",
        ha="right",
        fontsize=8.5,
        color=DIET_STYLE[diet]["color"],
    )

# Panel B: the land-use response -- cropland contracts, spared land grows.
for diet, st in DIET_STYLE.items():
    keys = ["ref", *FRONT]
    sc = [df.loc[(diet, s), "water_scarcity"] for s in [REF[diet], *FRONT]]
    dcrop = [
        lands[(diet, k)]["cropland (Mha)"] - lands[(diet, "ref")]["cropland (Mha)"]
        for k in keys
    ]
    dspar = [
        lands[(diet, k)]["spared land (Mha)"]
        - lands[(diet, "ref")]["spared land (Mha)"]
        for k in keys
    ]
    ax_b.plot(
        sc,
        dcrop,
        color=st["color"],
        marker=st["marker"],
        ls="-",
        label=f"{st['label']}: cropland",
    )
    ax_b.plot(
        sc,
        dspar,
        color=st["color"],
        marker=st["marker"],
        ls=":",
        label=f"{st['label']}: spared land",
    )
ax_b.set_xscale("log")
ax_b.invert_xaxis()
ax_b.axhline(0, color="k", lw=0.8)
ax_b.set(
    xlabel="Total water scarcity (Mm3 world-eq, millions)",
    ylabel="Change vs reference (Mha)",
    title="B. Cropland contracts; spared land grows",
)
ax_b.legend(frameon=False, fontsize=8)

fig5.tight_layout()
plt.show()

# Land-use summary table along the front.
land_tbl = pd.DataFrame(
    {(diet, k): lands[(diet, k)] for diet in REGIMES for k in ["ref", *FRONT]}
).T.round(1)
land_tbl

**The footprint never expands.** Panel B is the crux: as water scarcity is priced
down, total cropland *shrinks* (about -32 Mha fixed, -35 Mha flexible at wp3) and
*spared* land *grows* by a similar amount. The irrigated area that is given up is
essentially **not** re-created as rainfed area -- per-crop, the drop in irrigated area
equals the drop in *total* area for that crop, while global mean yield barely moves
(~3.04 -> 3.03 Mt/Mha). So the abandoned water-intensive production is simply not
replaced on new land; it is absorbed by *less output*, not by *more area*. Land
conversion (deforestation) therefore barely moves (+3.5 MtCO2e flexible, and a still
modest +20 fixed), and the freed land earns regrowth credits, so net sequestration
*increases*.

**The savings are demand- and intensity-side, not land-side.** Panel A shows the drop
is dominated by the highest-GWP activities, because *water-scarce* production and
*emission-intensive* production substantially overlap:

* **Rice** is the single largest irrigated crop (~73 Mha irrigated) and is pure CH4 --
  pricing water cuts paddy irrigation and rice methane together (-24 to -28 MtCO2e).
* **Irrigated feed** (maize, silage-maize, alfalfa) underpins livestock, so trimming it
  pulls down enteric fermentation, manure CH4/N2O and feed-crop fertilizer N2O.
* A water price thus acts partly as a *proxy carbon price* on exactly the methane- and
  nitrous-oxide-heavy parts of the system, plus an extra sequestration credit from the
  spared land.

**Why the flexible diet gains ~5x more (-193 vs -41 MtCO2e).** Under a fixed diet the
only freedom is production-side: trim feed/over-production and shed marginal irrigation,
which yields modest gross savings (rice, fertilizer, enteric) that are partly offset by
the small rise in land-use-change emissions. Under a flexible diet the water-induced cost
increase additionally propagates to food prices, and consumption shifts *off the
thirsty, emission-dense foods* -- grain/rice (-44 Mt), sugar (-20 Mt) and animal products
(dairy, poultry) -- toward cheaper, low-water, low-emission calories (vegetable oil
+22 Mt, whole grains). That demand contraction is what turns a modest production-side
trim into a large, broad-based emissions cut, with the bulk landing on enteric
fermentation (-58), manure (-24), rice (-28), fertilizer (-22) and extra sequestration
(-54).

## Does the co-benefit survive without the 2020-production anchor?

Every front above was computed with GLADE's calibrated **deviation penalties**
switched on -- L1 costs that hold cropland, grassland and feed near their
(inefficient) 2020 patterns. This raises a worry: the emissions co-benefit of
water pricing might be an *artefact* of that anchor. If production cannot move,
the model cannot respond to a water price by expanding onto new (forest) land, so
the very deforestation channel that would *raise* emissions is suppressed by
construction.

To test this we re-ran the entire water front with the penalties OFF
(`deviation_penalty.enabled = false`; the `*nd` scenarios). Production is then
free to reallocate, limited only by the per-crop and per-animal growth caps. For
the flexible diet the consumer values are re-calibrated against a dedicated
penalty-free baseline (`baselinend`), so revealed preferences are measured
against efficient -- not anchored -- production.

In [ ]:
# === Water pricing with vs without the 2020-production anchor (deviation penalties) ===
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CH4_GWP, N2O_GWP = 27.0, 273.0
WP_TAIL = ["0003", "001", "003", "01", "03", "1", "3", "10", "30", "100", "300", "1000"]


def _front_row(config, scen, scen_defs):
    n = pypsa.Network(
        str(ROOT / "results" / config / "solved" / f"model_scen-{scen}.nc")
    )
    e = n.stores.dynamic.e.iloc[0]
    links, p0 = n.links.static, n.links.dynamic.p0.iloc[0]
    cp = links[links["carrier"].isin(["crop_production", "crop_production_multi"])]
    area = p0.reindex(cp.index).clip(lower=0)

    def pick(mask):
        return float(p0.reindex(links.index[mask]).clip(lower=0).sum())

    wp, _ = lever_prices(scen_defs, scen)
    em = extract_net_emissions(n, CH4_GWP, N2O_GWP).groupby("source")["mtco2eq"].sum()
    return {
        "water_price": wp,
        "water_scarcity": float(e["store:impact:water_scarcity"]) / 1e6,
        "emissions": float(e["store:emission:ghg"]),
        "cropland": float(area.sum()),
        "spared": pick(
            links["carrier"].isin(["spare_land", "spare_existing_grassland"])
        ),
        "luc": float(em.get("Land Use Change", 0.0)),
        "seq": float(em.get("Carbon sequestration", 0.0)),
        "enteric": float(em.get("Enteric fermentation", 0.0)),
        "rice": float(em.get("Rice cultivation", 0.0)),
    }


series = {}
for diet, spec in REGIMES.items():
    defs = {
        k: v
        for k, v in yaml.safe_load(
            (ROOT / "config" / f"{spec['config']}.yaml").read_text()
        )["scenarios"].items()
        if v is not None
    }
    for pen, suf in [("dev", ""), ("nodev", "nd")]:
        names = [spec["reference"] + suf, *[f"wp{w}{suf}" for w in WP_TAIL]]
        s = pd.DataFrame([_front_row(spec["config"], nm, defs) for nm in names])
        ref_em = s.loc[s.water_price == 0, "emissions"].iloc[0]
        s["d_emissions"] = s["emissions"] - ref_em
        series[(diet, pen)] = s.sort_values("water_scarcity")

STYLE4 = {
    ("fixed", "dev"): {
        "color": "#1f77b4",
        "ls": "-",
        "marker": "o",
        "label": "fixed, with penalty",
    },
    ("fixed", "nodev"): {
        "color": "#1f77b4",
        "ls": "--",
        "marker": "x",
        "label": "fixed, no penalty",
    },
    ("flexible", "dev"): {
        "color": "#d62728",
        "ls": "-",
        "marker": "s",
        "label": "flexible, with penalty",
    },
    ("flexible", "nodev"): {
        "color": "#d62728",
        "ls": "--",
        "marker": "x",
        "label": "flexible, no penalty",
    },
}

fig6, ax = plt.subplots(2, 2, figsize=(13, 9))
for key, st in STYLE4.items():
    s = series[key]
    ln = {"color": st["color"], "ls": st["ls"], "marker": st["marker"], "ms": 5}
    ax[0, 0].plot(s.water_scarcity, s.emissions, label=st["label"], **ln)
    ax[0, 1].plot(s.water_scarcity, s.d_emissions, label=st["label"], **ln)
    ax[1, 0].plot(s.water_scarcity, s.luc, label=st["label"], **ln)
    ax[1, 1].plot(s.water_scarcity, s.spared, label=st["label"], **ln)

for a in ax.flat:
    a.set_xscale("log")
    a.invert_xaxis()
    a.set_xlabel("Total water scarcity (Mm3 world-eq, millions)  -- reduced -->")
ax[0, 0].axhline(0, color="k", lw=0.8)
ax[0, 0].set(ylabel="Net GHG emissions (MtCO2e)", title="A. Net emissions (absolute)")
ax[0, 0].legend(frameon=False, fontsize=8)
ax[0, 1].axhline(0, color="k", lw=0.8)
ax[0, 1].set(
    ylabel="Change vs own unpriced ref (MtCO2e)",
    title="B. Effect of the water lever  (>0 = trade-off, <0 = co-benefit)",
)
ax[1, 0].set(
    ylabel="Land-use-change emissions (MtCO2e)",
    title="C. Deforestation pulse from water pricing",
)
ax[1, 1].set(ylabel="Spared land (Mha)", title="D. Land sparing")
fig6.suptitle(
    "Water pricing with vs without the 2020-production anchor (deviation penalties)",
    fontsize=13,
    y=0.99,
)
fig6.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# Unpriced reference points and the change at wp3.
COLS = ["emissions", "cropland", "spared", "seq", "luc"]
ref_tbl = pd.DataFrame(
    {k: series[k].loc[series[k].water_price == 0].iloc[0][COLS] for k in STYLE4}
).T
print("Unpriced reference points (MtCO2e / Mha):")
print(ref_tbl.round(0))


def _at_wp3(k):
    s = series[k]
    r0 = s.loc[s.water_price == 0].iloc[0]
    r3 = s.loc[s.water_price == 0.3].iloc[0]
    return (r3 - r0)[
        ["emissions", "luc", "seq", "enteric", "rice", "cropland", "spared"]
    ]


print("\nChange at wp3 vs own unpriced reference (MtCO2e / Mha):")
print(pd.DataFrame({k: _at_wp3(k) for k in STYLE4}).T.round(0))

**Removing the anchor changes the baseline far more than any water price does.**
At zero water price, dropping the penalties lets the model abandon today's
dispersed production and concentrate on the best land: cropland falls ~30%
(1417 -> ~985 Mha) and pasture ~50% (2284 -> ~1140 Mha), freeing ~1590 Mha that
regrows. Carbon sequestration jumps from -0.93 to -6.4 GtCO2e and *net*
food-system emissions flip from +5.5 GtCO2e to a **-0.67 GtCO2e sink** (panel A:
the gap between the solid and dashed baselines on the left). This is the familiar
land-sparing result -- and exactly why the penalties exist: this unconstrained
optimum is not a realistic depiction of how production is actually distributed.

**On that sink baseline, pricing water genuinely reallocates land, and a
deforestation pulse appears in *both* diets** (panel C): land-use-change
emissions climb from ~335 MtCO2e to ~950 (fixed) / ~580 (flexible) as irrigation
is squeezed out. This is precisely the channel the penalised fronts had hidden --
there, LUC stayed flat at ~45 MtCO2e regardless of the water price.

**Fixed diet -- the trade-off the intuition predicted** (panel B, blue dashed
*above* zero). With demand locked, lost irrigated output must be replaced by
land-hungry rainfed production, so cropland and pasture re-expand, ~150 Mha of
sparing is handed back, sequestration weakens (+0.6 GtCO2e) and LUC rises
(+0.6 GtCO2e). Net emissions *rise* +259 MtCO2e at wp3 and +1216 at the extreme,
turning the -0.67 GtCO2e sink back into a +0.54 GtCO2e source. Reducing water
scarcity now *costs* carbon.

**Flexible diet -- the co-benefit survives, but it is now a contest** (panel B,
red dashed *below* zero). The diet still shifts off thirsty, emission-dense foods
(enteric -125, rice -39 MtCO2e at wp3) and those demand-side cuts *outweigh* the
deforestation pulse (LUC +82) -- net -93 MtCO2e at wp3. But that is roughly half
the penalised co-benefit (-93 vs -193) precisely because the land-reallocation
channel is now active and pushing the other way, and it saturates at high prices
(panel B flattens) as LUC keeps climbing.

**Conclusion.** Yes -- once today's production patterns are no longer enforced, a
meaningful irrigation <-> emissions trade-off appears. It is an unambiguous
trade-off under a fixed diet (pricing water drives deforestation and re-emits
stored carbon) and a weakened, contested co-benefit under a flexible diet (diet
shifts still win at moderate prices). The strong co-benefit of the penalised
analysis was substantially an artefact of the fixed-production anchor. The honest
reading is that whether water and climate goals align depends on how freely
production can reallocate and on whether diets can adapt -- not on the water price
alone.

## Takeaways

* The non-marginal, tiered AWARE supply turns water scarcity into a first-class
  optimisation objective. A merit-order regularizer ensures the unpriced baseline
  draws the least-scarce water available, rather than an arbitrary (degenerate)
  high-CF tier mix.
* **There is no free water-scarcity reduction.** The unpriced baseline is already
  merit-ordered (~3.4); cutting scarcity below it requires real prices and follows a
  convex cost tail, driven by genuinely using less irrigation and shifting production --
  where production-stability penalties and real trade-offs bind. (An earlier "~75% cut
  at near-zero cost" was an artifact of an unpriced tie-break, now removed.)
* **Eliminating irrigation is asymptotic and very expensive.** Driving withdrawal toward
  zero never hits a clean cutoff; near-total elimination (~99.5%) costs several hundred
  billion USD/yr above baseline, with no hard infeasibility -- the fixed 2020 diet can run
  almost entirely rainfed, just at extreme cost.
* **Diet flexibility does little for water scarcity at moderate prices, but pays off at
  the extreme.** Both regimes share the same baseline and near-identical fronts where
  prices are modest; only when pushing irrigation toward zero does a flexible diet pull
  ahead -- reaching lower irrigation and eliminating it ~25% more cheaply by shifting off
  the thirstiest foods.
* **Water and climate goals stay complementary**, and the water->emissions co-benefit is
  several times larger under a flexible diet (foods that are thirsty are often also
  emission-intensive).

### Caveats

* AWARE characterises **blue-water** consumption only; rainfed ("green water")
  production carries no scarcity, part of why shifting irrigated -> rainfed lowers the
  metric.
* **Water is pooled per model region** (~10+ AWARE basins each), so intra-region
  inter-basin substitution is frictionless. This sets how low the merit-order baseline
  can go; finer regions would tighten the pooling and could raise the achievable-scarcity
  floor.
* Irrigated *area* floors near ~100 Mha even as water *draw* approaches zero at extreme
  prices (crops stay irrigation-classified but draw negligible water); the
  scarcity-relevant metric is withdrawn volume.
* The water price is a policy/shadow lever, not a market price; the "cost" axis is real
  system cost with that lever removed, and is only compared within a diet regime.

* **The co-benefit is partly an artefact of the production anchor.** With the calibrated deviation penalties OFF, the unpriced baseline already spares ~1590 Mha and is a net carbon *sink* (-0.67 vs +5.5 GtCO2e). On that baseline, pricing water drives real land reallocation and deforestation: under a *fixed* diet this becomes a genuine irrigation<->emissions *trade-off* (+0.26 GtCO2e at wp3, up to +1.2 at the extreme); under a *flexible* diet the diet-shift co-benefit survives but is roughly halved (-93 vs -193 MtCO2e at wp3) as deforestation eats into it.
